In [15]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = 'mps' if torch.mps.is_available() else 'cpu'
device

'mps'

In [16]:
# EfficientNet model expects larger images, so we need to resize the images to 224x224 pixels. We also normalize the images using the mean and standard deviation of the ImageNet dataset.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_data = datasets.ImageFolder(
    root="./cifar10/train",
    transform=transform
)

test_data = datasets.ImageFolder(
    root="./cifar10/test",
    transform=transform
)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

# Load pretrained EfficientNet-B0, then replace its 1,000-class output layer
model = models.efficientnet_b0(weights="DEFAULT")

# Keep all EfficientNet convolutional layers.
# Replace only the original 1000-class ImageNet output layer.
n_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(n_features, 128),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(128, 10)
)
model = model.to(device)

In [17]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [18]:
def iterate_efficientnet(model, x):
    x = model.features[0](x)
    print("After features[0]:", x.shape)

    x = model.features[1](x)
    print("After features[1]:", x.shape)

    x = model.features[2](x)
    print("After features[2]:", x.shape)

    return x

In [ ]:
images, labels = next(iter(test_loader))
image = images[0].unsqueeze(0)  # Shape: [1, 3, 224, 224]

model.eval()
with torch.no_grad():
    features = iterate_efficientnet(model, image.to(device))

After features[0]: torch.Size([1, 32, 112, 112])
After features[1]: torch.Size([1, 16, 112, 112])
After features[2]: torch.Size([1, 24, 56, 56])


In [20]:
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        predictions = model(images)
        loss = loss_fn(predictions, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_loader):.4f}")

KeyboardInterrupt: 

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        predicted = outputs.argmax(dim=1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test accuracy: {accuracy:.2f}%")

Test accuracy: 88.82%
